# BTGenBot-2 Evaluation Server

This notebook serves the [BTGenBot-2](https://huggingface.co/AIRLab-POLIMI/llama-3.2-1b-it-ft-lora-bt) model via an HTTP API for thesis evaluation.

**Instructions:**
1. Runtime → Change runtime type → T4 GPU
2. Run all cells in order
3. Copy the ngrok URL printed at the end
4. Paste it into your evaluation runner as `--colab-url`

The server stays alive until the Colab runtime is disconnected (typically 90 min idle, ~12h active).

In [ ]:
# Install dependencies (run once)
!pip install -q transformers accelerate peft torch pyngrok uvicorn fastapi nest-asyncio

In [ ]:
import os
import json
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
import uvicorn
import nest_asyncio
from pyngrok import ngrok
import gc

nest_asyncio.apply()

In [ ]:
# --- Configuration ---
BASE_MODEL = "meta-llama/Llama-3.2-1B-Instruct"
LORA_MODEL = "AIRLab-POLIMI/llama-3.2-1b-it-ft-lora-bt"

# You MUST set a HuggingFace token with access to both models.
# The base model is gated (Llama). Get a token from https://huggingface.co/settings/tokens
HF_TOKEN = os.environ.get("HF_TOKEN", "")

# Ngrok auth token (get free from https://dashboard.ngrok.com/get-started/your-authtoken)
NGROK_TOKEN = os.environ.get("NGROK_TOKEN", "")

In [ ]:
# --- Load model with LoRA adapter ---
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL,
    token=HF_TOKEN if HF_TOKEN else None
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Loading base model (this takes ~1 minute on T4)...")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    device_map="auto",
    token=HF_TOKEN if HF_TOKEN else None
)

print("Loading LoRA adapter...")
model = PeftModel.from_pretrained(base_model, LORA_MODEL)
model = model.merge_and_unload()  # merge for faster inference
model.eval()

# Free memory
del base_model
gc.collect()
torch.cuda.empty_cache()

print(f"Model loaded. VRAM used: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

In [ ]:
# --- Inference function using chat template ---
def generate_bt(task: str, actions: str, max_new_tokens: int = 2048) -> str:
    """
    Generate a Behavior Tree from a task description and action list.
    Follows BTGenBot-2's expected input format.
    """
    input_text = f"Task:\n{task}\n\nActions:\n{actions}"
    
    messages = [
        {"role": "system", "content": "You are a Behavior Tree generator. You output valid BehaviorTree.CPP XML only."},
        {"role": "user", "content": input_text}
    ]
    
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.0,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )
    
    # Decode only the new tokens
    generated = outputs[0][inputs.input_ids.shape[1]:]
    result = tokenizer.decode(generated, skip_special_tokens=True)
    
    # Clean up the output - extract XML if surrounded by other text
    if "<root" in result:
        start = result.index("<root")
        if "</root>" in result:
            end = result.index("</root>") + len("</root>")
            result = result[start:end]
    
    return result.strip()

In [ ]:
# --- Set up FastAPI server ---

app = FastAPI(title="BTGenBot-2 Evaluation Server")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)

class GenerateRequest(BaseModel):
    task: str
    actions: str
    max_new_tokens: int = 2048

class GenerateResponse(BaseModel):
    xml: str
    model: str = "BTGenBot-2"

@app.get("/health")
def health():
    return {"status": "ok", "model": "BTGenBot-2", "vram_gb": round(torch.cuda.memory_allocated() / 1e9, 1)}

@app.post("/generate")
def generate(req: GenerateRequest):
    try:
        xml = generate_bt(req.task, req.actions, req.max_new_tokens)
        return GenerateResponse(xml=xml)
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

print("FastAPI app created.")

In [ ]:
# --- Start ngrok tunnel + uvicorn ---

if NGROK_TOKEN:
    ngrok.set_auth_token(NGROK_TOKEN)

# Open ngrok tunnel to port 8000
public_url = ngrok.connect(8000).public_url
print(f"\n{'='*60}")
print(f"🚀 BTGenBot-2 server is LIVE")
print(f"Public URL: {public_url}")
print(f"Health check: {public_url}/health")
print(f"Generate endpoint: {public_url}/generate")
print(f"\nCopy this URL for your evaluation runner:")
print(f"  python3 evaluation/scripts/run_evaluation.py --methods M2 --colab-url {public_url}")
print(f"{'='*60}\n")

# Start uvicorn server (blocking)
uvicorn.run(app, host="0.0.0.0", port=8000)